In [ ]:
# === M2 PySR Colab setup (= Track 3.4) ===
# PySR = Julia-backed symbolic regression、 TVT = f(GR, ANCC, Z, MD, formation) の閉形式公式 auto-discover.
# Runtime: A100 GPU 推奨、 30-90 min for 770 wells × 5 feature × 20 generations.

import os, sys, subprocess, getpass
from pathlib import Path

# 1) Install PySR (= Julia auto-install on first call)
!pip install -q pysr lightgbm scipy scikit-learn 2>&1 | tail -3

# 2) Kaggle auth (= ACCESS_TOKEN)
Path('/root/.kaggle').mkdir(parents=True, exist_ok=True)
TOKEN_FILE = Path('/root/.kaggle/access_token')
if not TOKEN_FILE.exists() or not TOKEN_FILE.read_text().strip().startswith('KGAT_'):
    token = getpass.getpass('Paste your Kaggle API token (= KGAT_...): ').strip()
    if not token.startswith('KGAT_'):
        raise RuntimeError(f'Token should start with KGAT_, got {token[:8]!r}')
    TOKEN_FILE.write_text(token)
    os.chmod(str(TOKEN_FILE), 0o600)
os.environ['KAGGLE_API_TOKEN'] = TOKEN_FILE.read_text().strip()
result = subprocess.run(['kaggle', 'competitions', 'list', '-s', 'rogii'], capture_output=True, text=True, env={**os.environ})
assert 'rogii' in result.stdout.lower(), 'auth failed'
print('Kaggle auth OK')

# 3) DL competition data
DATA_ROOT = Path('/content/rogii_data')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
if not (DATA_ROOT / 'competition').exists():
    !kaggle competitions download -c rogii-wellbore-geology-prediction -p /content/rogii_data
    !cd /content/rogii_data && unzip -q rogii-wellbore-geology-prediction.zip -d competition
print('Data ready')

# 4) Init PySR (= triggers Julia install on first import)
from pysr import PySRRegressor
print('PySR ready (= Julia backend installed)')


In [ ]:
# === Symbolic regression: discover TVT formula ===
import pandas as pd, numpy as np, os, random
from pathlib import Path
from pysr import PySRRegressor

TRAIN_DIR = Path('/content/rogii_data/competition/train')
files = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
print(f'Train wells: {len(files)}')

# Sample to keep PySR tractable (= 50K rows)
random.seed(42)
sample_wells = random.sample(files, min(100, len(files)))

rows = []
for f in sample_wells:
    df = pd.read_csv(f, usecols=['MD','X','Y','Z','ANCC','GR','TVT']).dropna()
    if len(df) > 0:
        rows.append(df.sample(min(500, len(df)), random_state=42))
data = pd.concat(rows, ignore_index=True).dropna()
print(f'Total rows: {len(data):,}')

# Features (= explanatory) and target
X = data[['MD', 'Z', 'GR', 'ANCC']].values
y = data['TVT'].values

# Standardize for stability
X_mean, X_std = X.mean(0), X.std(0)
X_norm = (X - X_mean) / X_std
y_mean, y_std = y.mean(), y.std()
y_norm = (y - y_mean) / y_std

# Symbolic regression
print('Running PySR (= ~30-60 min on A100)...')
model = PySRRegressor(
    niterations=40,
    populations=20,
    population_size=50,
    binary_operators=['+', '-', '*', '/'],
    unary_operators=['exp', 'log', 'sin', 'cos', 'square'],
    maxsize=20,
    extra_sympy_mappings={},
    elementwise_loss='loss(prediction, target) = (prediction - target)^2',
    progress=True,
    random_state=42,
    deterministic=True,
    procs=0,
)
model.fit(X_norm, y_norm)
print('\n=== Best equations ===')
print(model.equations_.head(10))


In [ ]:
# === Save best equations + upload as Kaggle dataset ===
eqs = model.equations_
eqs.to_csv('/content/pysr_equations.csv', index=False)
print('Saved /content/pysr_equations.csv')

# Display top 5 by score (= MSE × complexity penalty)
for _, row in eqs.head(5).iterrows():
    print(f'  complexity {row["complexity"]}: {row["equation"]} (loss {row["loss"]:.4f})')

print('\n=== Next: upload as Kaggle dataset ky7240/rogii-pysr-equations ===')
print('  mkdir -p /content/upload_pysr && cp /content/pysr_equations.csv /content/upload_pysr/')
print('  cd /content/upload_pysr')
print('  cat > dataset-metadata.json << EOF')
print('  {"title": "ROGII PySR Equations", "id": "ky7240/rogii-pysr-equations", "licenses": [{"name": "CC0-1.0"}]}')
print('  EOF')
print('  kaggle datasets create -p /content/upload_pysr')
